## Model Monitoring and Drift Detection

This notebook evaluates whether the feature distribution changes between the validation data used during model development and simulated incoming data.

The objective is to:

- detect feature drift,
- simulate a change in sensor behavior,
- identify potentially affected features,
- define a simple retraining trigger.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
X_val = pd.read_csv("X_val_imputed.csv")

print("Validation data shape:", X_val.shape)

Validation data shape: (4070, 64)


In [4]:
# Create a copy of the validation data
# This will represent simulated new incoming data

X_new = X_val.copy()

print("New data shape:", X_new.shape)

New data shape: (4070, 64)


In [5]:
drift_features = [
    "sensor_4_rolling_mean",
    "sensor_15_rolling_mean",
    "sensor_11_rolling_mean"
]

for feature in drift_features:
    X_new[feature] = X_new[feature] * 1.15

In [6]:
for feature in drift_features:
    print(f"\n{feature}")
    print("Original mean:", X_val[feature].mean())
    print("New mean:", X_new[feature].mean())


sensor_4_rolling_mean
Original mean: 1408.895426208026
New mean: 1620.2297401392302

sensor_15_rolling_mean
Original mean: 8.441621223996723
New mean: 9.707864407596231

sensor_11_rolling_mean
Original mean: 47.5394595004095
New mean: 54.67037842547092


In [7]:
from scipy.stats import ks_2samp

for feature in drift_features:
    statistic, p_value = ks_2samp(
        X_val[feature],
        X_new[feature]
    )

    print(f"\n{feature}")
    print("KS statistic:", statistic)
    print("p-value:", p_value)


sensor_4_rolling_mean
KS statistic: 1.0
p-value: 0.0

sensor_15_rolling_mean
KS statistic: 1.0
p-value: 0.0

sensor_11_rolling_mean
KS statistic: 1.0
p-value: 0.0


In [8]:
drift_results = []

for feature in drift_features:
    statistic, p_value = ks_2samp(
        X_val[feature],
        X_new[feature]
    )

    drift_results.append({
        "feature": feature,
        "ks_statistic": statistic,
        "p_value": p_value,
        "drift_detected": p_value < 0.05
    })

drift_results = pd.DataFrame(drift_results)

drift_results

,feature,ks_statistic,p_value,drift_detected
0,sensor_4_rolling_mean,1.0,0.0,True
1,sensor_15_rolling_mean,1.0,0.0,True
2,sensor_11_rolling_mean,1.0,0.0,True


In [10]:
# Summary of drift detection results

total_features = len(drift_results)
drifted_features = drift_results["drift_detected"].sum()
drift_threshold = 0.05

print("Drift Monitoring Summary")
print("------------------------")
print("Monitored features:", total_features)
print("Drifted features:", drifted_features)
print("Drift threshold (p-value):", drift_threshold)

if drifted_features > 0:
    print("Retraining required: Yes")
else:
    print("Retraining required: No")

Drift Monitoring Summary
------------------------
Monitored features: 3
Drifted features: 3
Drift threshold (p-value): 0.05
Retraining required: Yes


In [9]:
if drift_results["drift_detected"].any():
    print("⚠️ DATA DRIFT DETECTED")
    print("Retraining trigger activated.")
else:
    print("✅ No significant data drift detected.")
    print("Retraining not required.")

⚠️ DATA DRIFT DETECTED
Retraining trigger activated.


## Monitoring Conclusion

The simulated incoming data was created by introducing a 15% shift in three monitored sensor-based features.

The Kolmogorov-Smirnov (KS) test detected statistically significant distribution changes in all three monitored features using a p-value threshold of 0.05.

Therefore, data drift was detected and the retraining trigger was activated.

> Note: The drift in this notebook is intentionally simulated to demonstrate how the monitoring and retraining mechanism works. It does not represent real production drift.